# Cross-Modal Faithfulness Evaluation Showcase

This notebook runs the structured-trace faithfulness-evaluation pipeline with real models. The target model generates typed scientific support traces, the intervention builder creates original, deterministic wrong-binding, wrong-principle, wrong-inference, delete-binding, and paraphrase-control traces, the judge provides secondary diagnostics, and the target model re-answers so behavioral effects can be measured.

Start with a small image-bearing ScienceQA run. Increase `MAX_EXAMPLES` only after the small run completes successfully.


## Pipeline Components Used Outside This Notebook

This notebook is a showcase, not the full implementation. The experiment depends on package modules and files that are kept outside the notebook so they can be tested, reused, and changed without notebook-only logic.

- `aria.loaders`: ScienceQA, AI2D, and MMMU-Pro loading, image materialization, and pilot filters for image/explanation/diagram-heavy examples.
- `aria.schemas`: typed support traces with visual evidence, textual evidence, binding claim, scientific principle, inference, conclusion, and final answer.
- `aria.interventions`: intervention metadata, validation, and conversion of intervention-model outputs into auditable `IntervenedTrace` records.
- `aria.metrics`: judge diagnostics plus target behavioral effects, retargeted-support following, corrupted-but-still-correct rate, necessity/sufficiency scores, shortcut index, and judge-effect correlation.
- `aria.pipeline`: orchestration for trace generation, interventions, judging, target re-answering, checkpoint/resume behavior, and artifact saving.
- `tests/`: unit and integration tests for schemas, parsers, interventions, metrics, and pipeline artifacts.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "aria").exists():
    ROOT = Path.cwd().parents[0]
sys.path.insert(0, str(ROOT))

ROOT

## Run the Real-Model Faithfulness Evaluation

This cell loads the configured trace generator, generates typed support traces, unloads it, applies faithfulness-evaluation interventions, judges traces as a secondary diagnostic, re-runs the target answerer, and saves JSONL/CSV artifacts.


In [ ]:
from aria.hf_models import DEFAULT_INTERVENTION_MODEL_ID, DEFAULT_TARGET_MODEL_ID, HFGenerationConfig, InterventionModel, TargetRerunModel, TraceGeneratorModel
from aria.loaders import load_scienceqa
from aria.pipeline import run_faithfulness_evaluation

MAX_EXAMPLES = 1
LOAD_IN_4BIT = True
TARGET_QUANTIZATION = "bnb4"
INTERVENTION_QUANTIZATION = "bnb4"
REQUIRE_GPU = True
MIN_VISUAL_TOKENS = 256
MAX_VISUAL_TOKENS = 768
RUN_TARGET_RERUN = True
TARGET_MODEL_ID = DEFAULT_TARGET_MODEL_ID
INTERVENTION_MODEL_ID = DEFAULT_INTERVENTION_MODEL_ID

output_dir = ROOT / "results" / "faithfulness_evaluation_scienceqa_notebook"
examples = load_scienceqa(
    max_examples=MAX_EXAMPLES,
    split="validation",
    image_dir=ROOT / "data" / "processed" / "scienceqa_images" / "validation",
    require_image=True,
    require_explanation=True,
)

config = {
    "experiment": "cross_modal_faithfulness_evaluation",
    "dataset": "scienceqa",
    "target_model": TARGET_MODEL_ID,
    "intervention_model": INTERVENTION_MODEL_ID,
    "max_examples": MAX_EXAMPLES,
    "run_target_rerun": RUN_TARGET_RERUN,
    "target_quantization": TARGET_QUANTIZATION,
    "intervention_quantization": INTERVENTION_QUANTIZATION,
    "require_gpu": REQUIRE_GPU,
    "min_visual_tokens": MIN_VISUAL_TOKENS,
    "max_visual_tokens": MAX_VISUAL_TOKENS,
}

trace_generator = TraceGeneratorModel(
    model_id=TARGET_MODEL_ID,
    load_in_4bit=LOAD_IN_4BIT,
    quantization=TARGET_QUANTIZATION,
    require_gpu=REQUIRE_GPU,
    min_visual_tokens=MIN_VISUAL_TOKENS,
    max_visual_tokens=MAX_VISUAL_TOKENS,
    generation=HFGenerationConfig(max_new_tokens=512),
)
intervention_model = InterventionModel(
    model_id=INTERVENTION_MODEL_ID,
    load_in_4bit=LOAD_IN_4BIT,
    quantization=INTERVENTION_QUANTIZATION,
    require_gpu=REQUIRE_GPU,
    generation=HFGenerationConfig(max_new_tokens=1536),
)
target_evaluator = (
    TargetRerunModel(
        model_id=TARGET_MODEL_ID,
        load_in_4bit=LOAD_IN_4BIT,
        quantization=TARGET_QUANTIZATION,
        require_gpu=REQUIRE_GPU,
        min_visual_tokens=MIN_VISUAL_TOKENS,
        max_visual_tokens=MAX_VISUAL_TOKENS,
        generation=HFGenerationConfig(max_new_tokens=16),
    )
    if RUN_TARGET_RERUN
    else None
)

summary = run_faithfulness_evaluation(
    examples=examples,
    output_dir=output_dir,
    config=config,
    trace_generator=trace_generator,
    intervention_generator=intervention_model,
    target_evaluator=target_evaluator,
    run_target_rerun=RUN_TARGET_RERUN,
)
summary


## Validate Canonical JSONL Artifacts

The parser may repair minor formatting around model JSON, such as Markdown code fences or trailing text, but it does not semantically rewrite reasoning. After parsing, traces must validate against the Pydantic schema.


In [ ]:
from aria.io import read_jsonl
from aria.parsing import parse_structured_trace
from aria.schemas import GeneratedTraceRecord, InterventionModelOutput, IntervenedTrace, JudgeOutput, TargetOutput

generated = read_jsonl(output_dir / "original_traces.jsonl", GeneratedTraceRecord)
intervened = read_jsonl(output_dir / "intervened_traces.jsonl", IntervenedTrace)
intervention_model_outputs = read_jsonl(output_dir / "intervention_model_outputs.jsonl", InterventionModelOutput)
judge_path = output_dir / "judge_outputs.jsonl"
judge_outputs = read_jsonl(judge_path, JudgeOutput) if judge_path.exists() else []
target_output_path = output_dir / "target_outputs.jsonl"
target_outputs = read_jsonl(target_output_path, TargetOutput) if target_output_path.exists() else []

{
    "original_traces": len(generated),
    "intervened_traces": len(intervened),
    "intervention_model_outputs": len(intervention_model_outputs),
    "judge_outputs": len(judge_outputs),
    "target_outputs": len(target_outputs),
    "first_trace_answer": generated[0].trace.final_answer,
}


In [ ]:
raw_model_text = '''```json
{
  "visual_evidence": [{"modality": "image", "content": "Droplets are visible on the cold glass.", "provenance": "image"}],
  "textual_evidence": [{"modality": "text", "content": "The question asks what happens to water vapor on a cold surface.", "provenance": "question"}],
  "binding_claim": "The visible droplets are bound to the cold-surface condition described in the question.",
  "scientific_principle": "Condensation changes gas to liquid on a cold surface.",
  "inference": "The vapor becomes liquid droplets.",
  "conclusion": "The reasoning supports answer B.",
  "final_answer": "B"
}
```
extra text
'''

parse_structured_trace(raw_model_text).model_dump()


## Inspect Example, Original Trace, and Intervened Trace

In [ ]:
example = examples[0]
original_trace = generated[0].trace
example.model_dump(), original_trace.model_dump()

In [ ]:
records_for_first = [record for record in intervened if record.example_id == example.example_id]
[
    {
        "intervention_id": record.intervention_id,
        "type": record.intervention.intervention_type,
        "target_component": record.intervention.target_component,
        "expected_flawed_component": record.expected_flawed_component,
        "final_answer": record.intervened_trace.final_answer,
    }
    for record in records_for_first
]

## Summary Tables

These are the first tables to inspect after a pilot: target behavioral mediation effects, overall judge behavior, and which intervention categories are behaviorally load-bearing or only judged as flawed.


In [ ]:
import pandas as pd

summary_df = pd.read_csv(output_dir / "metrics_summary.csv")
by_intervention = pd.read_csv(output_dir / "metrics_by_intervention.csv")
by_component = pd.read_csv(output_dir / "metrics_by_component.csv")
target_effects = pd.read_csv(output_dir / "target_effects.csv") if (output_dir / "target_effects.csv").exists() else pd.DataFrame()
mediation_metrics = pd.read_csv(output_dir / "mediation_metrics.csv") if (output_dir / "mediation_metrics.csv").exists() else pd.DataFrame()

summary_df, mediation_metrics, target_effects.head()


In [ ]:
by_intervention.sort_values("detection_accuracy")

In [ ]:
by_component.sort_values("localization_accuracy")

## Real-Model Notes

- `TraceGeneratorModel` prompts the target model to emit typed support traces directly.
- `InterventionModel` generates schema-constrained natural-language intervened traces. `JudgeModel` is optional for C2-Faith/FACT-E-style baselines.
- `TargetRerunModel` measures whether answer behavior changes or abstains (`n/a`) under original, intervened, and paraphrase-control traces.
- The current main behavioral endpoints are net effects relative to `paraphrase`: Net Flip Effect, Net Abstention Effect, Net Retarget Following Effect, and Net Correctness Drop.
- `no_trace`, `text_only_no_trace`, and `no_image_with_original_trace` are input-availability controls. They contextualize trace dependence but do not prove modality importance by themselves.
- The evaluation measures output-level interventional faithfulness proxies, not hidden internal causal mediation.